
<div style="border-radius:10px; border:1px solid #a3a3a3; padding:24px; background:linear-gradient(120deg, #ffffff 80%, #e7effc 100%); box-shadow:0 2px 12px 0 rgba(40,70,140,.07);">

<h2 style="color:#226db0; margin-top:0;">
  🔁 SLOWLY CHANGING DIMENSION TYPE 2 (SCD)
</h2>

<p style="font-size:1.13em; color:#233140; line-height:1.6em;">
  The <strong>NYCTAXI.SILVER.REGIONS_SCD_2</strong> table has been loaded using the SCD Type 2 methodology, ensuring auditability of region changes. Every update preserves historical data with <code>effective_start_date</code> and <code>effective_end_date</code> boundaries.
</p>

<ul style="font-size:1.04em; color:#333c4f;">
  <li><strong>Deduplication</strong> performed to ensure one entry per <code>id</code>, keeping the most recent.</li>
  <li><strong>Change detection</strong> identifies new or modified regions.</li>
  <li><strong>Automatic versioning</strong>: Previous records expired; new rows appended with updated data and timestamps.</li>
  <li>Each region’s lifecycle is transparently tracked for auditing and analytics.</li>
</ul>

<p style="margin-top:24px; color:#1b4475;"> 
  <em>Historical trends are never lost—every change in your regions is captured and retained.</em>
</p>

</div>

In [0]:
from pyspark.sql.functions import *

changed_regions_df = (spark.read.format('csv')
                        .option('header', 'true')
						.schema('id INT, name STRING, date STRING')
                        .load('/Volumes/nyctaxi/landing/regions')
			).withColumn('load_timestamp', current_timestamp())

changed_regions_df = changed_regions_df.withColumn(
    "date_ts", to_timestamp(col("date"), "dd-MM-yyyy")
)

changed_regions_df.orderBy('id').display()

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, current_timestamp, lit, to_timestamp
from pyspark.sql.types import TimestampType
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Parse source DATE column (dd-MM-yyyy → Timestamp)
source_df = changed_regions_df.withColumn(
    "date_ts", to_timestamp(col("date"), "dd-MM-yyyy")
)

# Deduplicate source: keep only the latest row per ID based on DATE
w = Window.partitionBy("id").orderBy(col("date_ts").desc())
deduped_source = (
    source_df
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .drop("rn")
)

# Load target Delta table
dt = DeltaTable.forName(spark, "NYCTAXI.SILVER.REGIONS_SCD_2")

# Current active rows
current_active = dt.toDF().filter(col("effective_end_date").isNull())

# Step 1: Find IDs that changed (new or updated)
changed_ids = (
    deduped_source.alias("s")
    .join(current_active.alias("t"), col("s.id") == col("t.id"), "left")
    .filter((col("t.id").isNull()) | (col("s.name") != col("t.name")))
    .select(col("s.id").alias("id"))
    .distinct()
)

# Step 2: Expire old records
dt.alias("t").merge(
    source=changed_ids.alias("c"),
    condition="t.id = c.id AND t.effective_end_date IS NULL"
).whenMatchedUpdate(
    set={"effective_end_date": current_timestamp()}
).execute()

# Step 3: Insert only new source rows
new_versions = (
    deduped_source.alias("s")
    .join(changed_ids.alias("c"), col("s.id") == col("c.id"), "inner")
    .select(
        col("s.id").alias("id"),
        col("s.name").alias("name"),
        col("s.date_ts").alias("effective_start_date")
    )
    .withColumn("effective_end_date", lit(None).cast(TimestampType()))
    .withColumn("load_timestamp", current_timestamp())
)

new_versions.write.mode("append").saveAsTable("NYCTAXI.SILVER.REGIONS_SCD_2")

In [0]:
dbutils.notebook.exit('SLOWLY CHANGINNG DIMENSION TYPE 2 IS SUCCESSFULLY LOADED INTO NYCTAXI.SILVER.REGIONS_SCD_2')